In [3]:
import pandas as pd
import glob
import os
import numpy as np
import shutil
from Granger_dictionaries import *

In [4]:
# Grand River
dir = "..\\data\\granger\\field\\GR\\2025\\Q3"
db = "GR"

qrt = "3Q25"
task_code = f"{db}_{qrt}"

idx = 2  # Change this index to select a different file

sampling_company_code = "STRATA"
# Get all Excel file paths in the directory
excel_paths = glob.glob(os.path.join(dir, "*.xlsx"))
# print(excel_paths)

EDD_NAME = excel_paths[idx].split("\\")[-1]
EDD_TO_PROCESS = pd.read_excel(f'{dir}\\{EDD_NAME}', header=None)
# Find coordinates of "Project No.:"
row_idx, col_pos = np.where(EDD_TO_PROCESS == "Project No.:")
# project_no = EDD_TO_PROCESS.iat[row_idx[0], col_pos[0] + 1]
# print(f"Project No.: {project_no}")

print(f"database: {db}")
print(f"quarter: {qrt}")
print(f"file: {EDD_NAME}")
EDD_TO_PROCESS.shape

database: GR
quarter: 3Q25
file: -1212 Q3 2025 SWL_Table 1.xlsx


(20, 7)

In [5]:
new_columns = ["#sys_loc_code", "measurement_date", "Measuring Point", "historical_reference_elev", "water_level_depth", "water_level_elev","remark"]
EDD = EDD_TO_PROCESS.copy()
# Keep the real data block and rename columns without using Series.rename()
EDD = EDD_TO_PROCESS.iloc[0:, :7].copy()
EDD.columns = new_columns

# Drop row if no date:
pattern = r"mw-|leachate|sw-|p-|pw|pz|surface|underdrain|cell|lc|electric|recycling|cooper|groesbeck|vmw-|lmw-"
EDD = EDD.dropna(subset=["measurement_date"])
EDD["#sys_loc_code"] = EDD["#sys_loc_code"].astype(str).str.strip()
EDD = EDD[EDD["#sys_loc_code"].str.lower().str.contains(pattern, na=False)]

# Map to standardized sample names if possible, else keep original code
# EDD['sys_loc_code'] = EDD['sys_loc_code'].map(sample_id_dict).fillna(EDD['sys_loc_code'])  
sample_id_dict_ci = {k.lower(): v for k, v in sample_id_dict.items()}
EDD['#sys_loc_code'] = (EDD['#sys_loc_code'].str.lower().map(sample_id_dict_ci)).fillna(EDD['#sys_loc_code'])

EDD['task_code'] = task_code
EDD['measurement_method'] = 'Tape'
EDD['reportable_yn'] = 'Y'
EDD['depth_unit'] = 'ft'

EDD['dip_or_elevation'] = np.where(EDD['#sys_loc_code'].str.contains(r'sw|SW|sg|SG'), 'Elevation', 'Dip')
EDD['dry_indicator_yn'] = np.where(EDD['water_level_depth'].notnull(), 'N', 'Y')

In [6]:
# 1. path to template
template_path = r"..\EDDTemplate\Field EDD Template.xlsx"
template_df = pd.read_excel(template_path, sheet_name="WaterLevel_v1")

EDD = EDD.reindex(columns=template_df.columns)
expected_columns = ["#sys_loc_code", "measurement_date", "equipment_code", "historical_reference_elev", "water_level_depth", "water_level_elev", "corrected_depth", "corrected_elev", "measured_depth_of_well", "depth_unit", "technician", "dry_indicator_yn", "measurement_method", "batch_number", "dip_or_elevation", "remark", "lnapl_cas_rn", "lnapl_depth", "dnapl_cas_rn", "dnapl_depth", "task_code", "approval_code", "custom_field_1", "custom_field_2", "custom_field_3", "custom_field_4", "TRC_Project_Number", "reportable_yn"]
try:
    assert EDD.columns.tolist() == expected_columns
    print("Column names match the expected list.")
except AssertionError:
    print("Column names do not match the expected list.")
    # The EDD to process has extra columns that are not in the expected list. Print the extra columns.
    print("Extra columns in EDD to process:", set(template_df.columns.tolist()) - set(expected_columns
    ))

# 2. Make an exact copy of the template file
# Ouptut path for the new EDD file to be created
# Make a unique name for the output file by including the original file name and a timestamp
timestamp = pd.Timestamp.now().strftime("%Y%m%d_%H%M%S")
output_path = f"{dir}\\{EDD_NAME.split('.')[0]}_FIELD_EDD_{timestamp}.xlsx"
shutil.copy(template_path, output_path)

# 3. Write your dataframe into the copied template
# (Using 'a' mode allows you to append/write data to an existing sheet)
with pd.ExcelWriter(
    output_path, engine="openpyxl", mode="a", if_sheet_exists="overlay"
) as writer:
    EDD.to_excel(
        writer,
        sheet_name="WaterLevel_v1",  # <-- Change this to the exact name of the sheet in your template
        index=False,
        header=False,  # <-- Set to False if your template already has the headers typed out
        startrow=2,  # <-- Starts writing on row 2 (0-indexed), assuming row 1 has your headers
    )

Column names match the expected list.


c:\ageglio-1\EDD Data Processor\dataprocessor\Lib\site-packages\openpyxl\worksheet\_reader.py:329: UserWarning: Data Validation extension is not supported and will be removed
  warn(msg)
c:\ageglio-1\EDD Data Processor\dataprocessor\Lib\site-packages\openpyxl\worksheet\_reader.py:329: UserWarning: Data Validation extension is not supported and will be removed
  warn(msg)
